# Travel Analytics Capstone — Google Colab Runner

**Purpose:** run the supplied project in Colab, train the flight-price regression and recorded-gender classification models, create artifacts, run tests, and smoke-test the API and hotel recommender.

**Before you start:** upload `travel-analytics-capstone.zip` when prompted. This notebook does not create a GitHub repository, deploy a real Kubernetes cluster, or run a Jenkins server; those require external services. Use the repository's Docker/Kubernetes/Airflow/Jenkins files for those stages and capture screenshots only after you actually run them.

**Project focus:** flight-price regression is the primary MLOps model. Gender classification is included as an academic exercise only; predictions about gender are sensitive and must not be used for consequential decisions.


In [ ]:
# 1. Upload the project ZIP file
from google.colab import files
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if not zip_names:
    raise ValueError("Please upload travel-analytics-capstone.zip")
ZIP_PATH = zip_names[0]
print("Uploaded:", ZIP_PATH)


In [ ]:
# 2. Extract project files
from pathlib import Path
import zipfile, shutil, os

WORK = Path("/content/travel-analytics-capstone")
if WORK.exists():
    shutil.rmtree(WORK)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall("/content/unpacked_project")

# The archive contains a top-level travel-analytics-capstone/ folder
candidate = Path("/content/unpacked_project/travel-analytics-capstone")
if not candidate.exists():
    raise FileNotFoundError("Expected folder travel-analytics-capstone inside the uploaded ZIP")
shutil.move(str(candidate), str(WORK))
os.chdir(WORK)
print("Project root:", WORK)
print("Files:", sorted(p.name for p in WORK.iterdir()))
print("Dataset files:", sorted(p.name for p in (WORK / "data").glob("*.csv")))


In [ ]:
# 3. Install project dependencies
# This can take a few minutes. Colab may show dependency resolver warnings; inspect any actual import/test errors below.
%pip install -q -r /content/travel-analytics-capstone/requirements.txt
%pip install -q pytest


In [ ]:
# 4. Inspect datasets
import pandas as pd
DATA = WORK / "data"
flights = pd.read_csv(DATA / "flights.csv")
hotels = pd.read_csv(DATA / "hotels.csv")
users = pd.read_csv(DATA / "users.csv")

print("flights:", flights.shape)
display(flights.head())
print("hotels:", hotels.shape)
display(hotels.head())
print("users:", users.shape)
display(users.head())
print("\nFlight columns:", flights.columns.tolist())
print("Hotel columns:", hotels.columns.tolist())
print("User columns:", users.columns.tolist())


## Modeling approach

- **Regression target:** `price` in `flights.csv`.
- **Candidate models:** Ridge regression and Random Forest regression.
- **Evaluation:** hold-out test split with MAE, RMSE, and R²; choose the candidate with the lower MAE.
- **Classification:** logistic regression to predict the dataset's recorded `gender` field from age/company, reported with accuracy, balanced accuracy, and macro-F1. This is a sensitive target and is included only to meet the academic task.
- Identifier-like fields such as `travelCode` and `userCode` are excluded from regression features to reduce leakage risk.


In [ ]:
# 5. Train models and log MLflow experiments
import os, sys, json
os.chdir(WORK)
sys.path.insert(0, str(WORK / "src"))
os.environ.setdefault("MLFLOW_TRACKING_URI", (WORK / "mlruns").as_uri())

from travel_analytics.train import train
results = train(data_dir=str(DATA), artifact_dir=str(WORK / "artifacts"))
print(json.dumps(results, indent=2))


In [ ]:
# 6. Review metrics and confirm model artifacts exist
from pathlib import Path
import json
ART = WORK / "artifacts"
for filename in ["flight_metrics.json", "gender_metrics.json"]:
    path = ART / filename
    print(f"--- {filename} ---")
    print(path.read_text())

print("\nArtifacts:")
for p in sorted(ART.iterdir()):
    print(p.name, f"({p.stat().st_size:,} bytes)")
assert (ART / "flight_price_model.joblib").exists()
assert (ART / "gender_classifier.joblib").exists()
print("\nModel artifacts verified.")


In [ ]:
# 7. Run automated project tests
%cd /content/travel-analytics-capstone
!PYTHONPATH=src pytest -q


In [ ]:
# 8. Smoke-test the Flask REST API without exposing a public endpoint
# Flask's test client runs requests in-process, so no tunnel or public URL is required.
import os, sys
os.environ["MODEL_DIR"] = str(ART)
os.environ["DATA_DIR"] = str(DATA)
sys.path.insert(0, str(WORK / "src"))

from travel_analytics.api import app
client = app.test_client()

health = client.get("/health")
print("GET /health:", health.status_code, health.get_json())
assert health.status_code == 200

# Use an actual row's valid feature values to construct a realistic regression request.
row = flights.iloc[0]
payload = {
    "from": str(row["from"]),
    "to": str(row["to"]),
    "flightType": str(row["flightType"]),
    "time": float(row["time"]),
    "distance": float(row["distance"]),
    "agency": str(row["agency"]),
    "date": str(row["date"]),
}
response = client.post("/predict/flight-price", json=payload)
print("POST /predict/flight-price:", response.status_code, response.get_json())
assert response.status_code == 200, response.get_data(as_text=True)


In [ ]:
# 9. Smoke-test hotel recommendations
from travel_analytics.recommend import recommend_hotels
first_place = str(hotels.iloc[0]["place"])
first_user_code = int(users.iloc[0]["code"]) if "code" in users.columns else None
recommendations = recommend_hotels(
    hotels=hotels,
    user_code=first_user_code,
    place=first_place,
    max_price=None,
    limit=5,
)
display(pd.DataFrame(recommendations))
assert isinstance(recommendations, list)
print(f"Recommendation smoke test returned {len(recommendations)} result(s) for place={first_place!r}.")


## MLOps components and what is/ isn't demonstrated in Colab

- **MLflow:** training logs candidate metrics and models to the local `mlruns/` tracking directory. You can inspect the run data in the next cell.
- **Flask:** the in-process test client verifies the health and prediction endpoints.
- **Streamlit:** source code is included in the repository. A local notebook runtime is temporary; for a real public demo, deploy to an appropriate hosting service and record the actual URL.
- **Docker:** the Dockerfile and Compose configuration are included; Colab generally does not provide a reliable Docker daemon by default.
- **Kubernetes:** YAML is included; apply it to a configured cluster, not merely to Colab.
- **Airflow:** the DAG is included; it must run in an Airflow environment with dependencies and paths configured.
- **Jenkins:** the Jenkinsfile is included; a configured Jenkins controller and credentials are required.

Do not claim those services are live unless you have run them and verified their outputs. Capture genuine screenshots from your own runs for the report.


In [ ]:
# 10. Check local MLflow tracking data
mlruns = WORK / "mlruns"
if mlruns.exists():
    print("MLflow local tracking directory:", mlruns)
    print("Top-level entries:", [p.name for p in mlruns.iterdir()][:20])
else:
    print("No local mlruns directory found; inspect MLflow tracking configuration and training output.")


In [ ]:
# 11. Package code, trained artifacts, and metrics for download
# Exclude large temporary caches; include source, datasets, deployment files, metrics, and trained model files.
import shutil, os
OUTPUT_ZIP = Path("/content/travel-analytics-capstone-completed.zip")
if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()

with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in WORK.rglob("*"):
        if not p.is_file():
            continue
        if any(part in {".pytest_cache", "__pycache__"} for part in p.parts):
            continue
        if p.name.endswith(".pyc"):
            continue
        zf.write(p, p.relative_to(WORK.parent))

print("Created:", OUTPUT_ZIP, "size:", f"{OUTPUT_ZIP.stat().st_size:,} bytes")
from google.colab import files
files.download(str(OUTPUT_ZIP))
